In [27]:
import happybase
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from datetime import datetime
from tqdm import tqdm
from pyspark.sql import Row
spark = (
    SparkSession.builder
    .appName("hive -> hbase (agg)")
    .master("spark://spark-master:7077")
    .enableHiveSupport()
    .config("hive.metastore.uris", "thrift://hive-metastore:9083")
    .config("spark.cores.max", "1")
    .config("spark.executor.cores", "1")
    .getOrCreate()
)

In [13]:
# Make sure the checkpoint table exists
spark.sql("""
CREATE TABLE IF NOT EXISTS cryptopredictions.batch_checkpoint (
    table_name STRING,
    last_processed_date DATE,
    run_ts TIMESTAMP
)
STORED AS PARQUET;
""")

25/12/17 01:42:05 WARN HiveClientImpl: Detected HiveConf hive.execution.engine is 'tez' and will be reset to 'mr' to disable useless hive logic


DataFrame[]

In [14]:
# Read checkpoint for your table
checkpoint_df = (
    spark.table("cryptopredictions.batch_checkpoint")
    .filter(F.col("table_name") == "cryptocurrencysnapshot")
)

# Get the latest checkpoint
latest_checkpoint_row = checkpoint_df.orderBy(F.col("last_processed_date").desc()).limit(1).collect()
print("Checkpoint:", latest_checkpoint_row)

if latest_checkpoint_row:
    checkpoint = latest_checkpoint_row[0]["last_processed_date"]
else:
    checkpoint = None  # No checkpoint yet

crypto_df = spark.table("cryptopredictions.cryptocurrencysnapshot")
if checkpoint:
    crypto_df = crypto_df.filter(F.col("PartitionDate") > checkpoint)
    if crypto_df.count() == 0:
        print("No new data")

Checkpoint: []


In [15]:
def aggregate_crypto(df, window_duration, granularity_label):
    return (
        df
        .groupBy(
            "Symbol",
            F.window("Datetime", window_duration).alias("w")
        )
        .agg(
            F.first("CurrentPrice").alias("open"),
            F.max("CurrentPrice").alias("high"),
            F.min("CurrentPrice").alias("low"),
            F.last("CurrentPrice").alias("close")
        )
        .withColumn("granularity", F.lit(granularity_label))
        .withColumn("timestamp", F.col("w.start"))
        .drop("w")
    )
    
crypto_1m  = aggregate_crypto(crypto_df, "1 minute", "1m")
crypto_10m = aggregate_crypto(crypto_df, "10 minutes", "10m")
crypto_1d  = aggregate_crypto(crypto_df, "1 day", "1d")

crypto_agg = crypto_1m.unionByName(crypto_10m).unionByName(crypto_1d)

crypto_agg.filter(F.col("granularity") == "1m").show(5)
crypto_agg.filter(F.col("granularity") == "10m").show(5)
crypto_agg.filter(F.col("granularity") == "1d").show(5)

+------+--------+--------+--------+--------+-----------+-------------------+
|Symbol|    open|    high|     low|   close|granularity|          timestamp|
+------+--------+--------+--------+--------+-----------+-------------------+
|   ETH|  3042.0| 3042.65|  3042.0| 3042.65|         1m|2025-11-20 00:45:00|
|   SOL|  143.41|  143.41|  143.34|  143.34|         1m|2025-11-20 09:00:00|
|   BTC|92319.97|92319.98|92314.02|92319.98|         1m|2025-11-20 08:35:00|
|   ETH| 3029.22| 3029.22|  3027.5|  3027.5|         1m|2025-11-20 09:15:00|
|   SOL|  143.89|  143.89|  143.58|  143.89|         1m|2025-11-20 03:35:00|
+------+--------+--------+--------+--------+-----------+-------------------+
only showing top 5 rows



+------+--------+--------+-------+--------+-----------+-------------------+
|Symbol|    open|    high|    low|   close|granularity|          timestamp|
+------+--------+--------+-------+--------+-----------+-------------------+
|   ETH|  3035.1|  3040.4|3024.14| 3035.33|        10m|2025-11-20 08:10:00|
|   ETH| 3043.81| 3050.36|3038.01| 3042.34|        10m|2025-11-20 03:30:00|
|   BTC|94189.39|94189.39|93720.0|93853.17|        10m|2025-11-17 17:10:00|
|   SOL|   130.3|  130.68|  130.3|  130.64|        10m|2025-12-14 18:50:00|
|   BTC|96225.26| 96355.9|96099.4|96147.71|        10m|2025-11-15 14:20:00|
+------+--------+--------+-------+--------+-----------+-------------------+
only showing top 5 rows



[Stage 7:==================================================>      (16 + 1) / 18]

+------+--------+--------+--------+-------+-----------+-------------------+
|Symbol|    open|    high|     low|  close|granularity|          timestamp|
+------+--------+--------+--------+-------+-----------+-------------------+
|   ETH| 3135.39| 3174.19| 2898.49|3061.38|         1d|2025-12-15 00:00:00|
|   BTC|88653.13|90472.39|87609.86|89325.5|         1d|2025-12-14 00:00:00|
|   ETH| 2794.72| 3031.78| 2783.07|2809.29|         1d|2025-12-02 00:00:00|
|   SOL|  132.65|  135.26|  123.78| 129.57|         1d|2025-12-15 00:00:00|
|   SOL|   132.6|  136.31|  127.79| 135.92|         1d|2025-12-07 00:00:00|
+------+--------+--------+--------+-------+-----------+-------------------+
only showing top 5 rows



In [ ]:
crypto_agg.filter(F.col("granularity") == "1d").orderBy('timestamp', ascending = False).show(10)

In [16]:
max_date = crypto_df.agg(F.max("PartitionDate")).collect()[0][0]
print(max_date)

[Stage 10:==============================================>         (15 + 1) / 18]

2025-12-15


In [29]:
def row_to_hbase(row):
    row_key = f"{row['Symbol']}#{row['timestamp']}#{row['granularity']}"
    return (
        row_key.encode(),
        {
            b"ohlc:open": str(row['open']).encode(),
            b"ohlc:high": str(row['high']).encode(),
            b"ohlc:low": str(row['low']).encode(),
            b"ohlc:close": str(row['close']).encode(),
        }
    )
      
# To HBase
run_ts = datetime.now()

connection = happybase.Connection(host='hbase')
table = connection.table('crypto_index_aggregates')
rows = crypto_agg.collect()
for row in tqdm(rows):
    key, data = row_to_hbase(row)
    table.put(key, data)

max_date = crypto_df.agg(F.max("PartitionDate")).collect()[0][0]

new_checkpoint = spark.createDataFrame([Row(
    table_name="cryptocurrencysnapshot",
    last_processed_date=max_date,
    run_ts=run_ts
)])

(new_checkpoint.write
    .mode("append")
    .format("hive")
    .saveAsTable("cryptopredictions.batch_checkpoint"))

print("HBase insertion and checkpoint update succeeded!")

100%|██████████| 158022/158022 [01:14<00:00, 2134.18it/s]
                                                                                

AnalysisException: The format of the existing table spark_catalog.cryptopredictions.batch_checkpoint is `HiveFileFormat`. It doesn't match the specified format `ParquetDataSourceV2`.

25/12/17 02:07:37 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


HBase insertion and checkpoint update succeeded!


In [32]:
spark.stop()